# 3. Modelling

**Goal:** predict which students sleep less than 7 hours a night (classification), and how many
hours they sleep (regression), from work, demographics, economy, health and mental health.

**Input:** `data/processed/clean_dataset_numeric.csv` from `01_cleaning.ipynb`, where every column is a number.

Sections: 3.1 setup, 3.2 data and targets, 3.3 which columns the models may use, 3.4 preprocessing pipeline.
The train/test split, baseline, models and evaluation follow in the next sections.

## 3.1 Setup

In [1]:
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import PROCESSED

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__)

RANDOM_STATE = 42  # used everywhere randomness is involved, so results can be reproduced

Python 3.14.7 | pandas 3.1.1 | numpy 2.5.4 | scikit-learn 1.9.1


## 3.2 Data and targets

- **Classification target:** `short_sleep` = 1 if the student sleeps less than 7 hours a night, else 0.
- **Regression target:** `sleep_hours_per_night`.

`short_sleep` is added in `01_cleaning.ipynb` (issue #9). Until that is merged, it is computed here
with the same definition, so this notebook runs either way.

In [2]:
df = pd.read_csv(PROCESSED / "clean_dataset_numeric.csv")

if "short_sleep" not in df.columns:  # remove this once 01_cleaning adds the column (#9)
    df["short_sleep"] = (df["sleep_hours_per_night"] < 7).astype(int)

print(df.shape)
print("Share with short sleep:", round(df["short_sleep"].mean(), 3))
df.head()

(1085, 23)
Share with short sleep: 0.232


,household_id,age,survey_weight,weekly_work_hours,sleep_hours_per_night,past_month_felt_rested,past_month_trouble_falling_asleep,past_month_trouble_staying_asleep,sex,activity_guideline_status,...,kids_household,education_level,family_income,food_security,housing_cost,high_bloodpressure,had_asthma,had_arthritis,had_diabetes,short_sleep
0,H068201,21,6729.923,30,8,4,2,1,2.0,3.0,...,1,5,14,1.0,0.0,0,0.0,0.0,0,0
1,H029511,28,13866.114,40,7,3,2,4,2.0,3.0,...,0,8,12,1.0,1.0,0,1.0,0.0,0,0
2,H065349,18,11149.228,0,8,2,2,2,2.0,3.0,...,3,1,5,1.0,0.0,0,0.0,0.0,0,0
3,H028264,19,16847.170,20,5,3,1,3,1.0,3.0,...,3,5,14,1.0,0.0,0,1.0,0.0,0,1
4,H008231,18,2690.076,40,7,3,2,3,2.0,2.0,...,0,4,6,1.0,0.0,0,0.0,0.0,0,0


## 3.3 Which columns the models may use

Some columns are left out because they would give the model the answer, or because they are not
properties of the student. Keeping them would make the model look better than it is (data leakage).

In [3]:
TARGET_CLASS = "short_sleep"
TARGET_REG = "sleep_hours_per_night"

EXCLUDED = {
    "short_sleep": "the classification target",
    "sleep_hours_per_night": "the regression target, and short_sleep is computed from it",
    "past_month_felt_rested": "a sleep outcome measured at the same time as the target",
    "past_month_trouble_falling_asleep": "a sleep outcome measured at the same time as the target",
    "past_month_trouble_staying_asleep": "a sleep outcome measured at the same time as the target",
    "survey_weight": "a survey weight for population estimates, not a property of the student",
    "household_id": "an id with no meaning",
}

# Nominal codes have no order, so they are one-hot encoded. Everything else is a count, a measure
# or an ordered scale, and is treated as a number.
CATEGORICAL = ["sex", "education_level"]
NUMERIC = [c for c in df.columns if c not in EXCLUDED and c not in CATEGORICAL]

pd.DataFrame({"reason left out": EXCLUDED})

,reason left out
short_sleep,the classification target
sleep_hours_per_night,"the regression target, and short_sleep is comp..."
past_month_felt_rested,a sleep outcome measured at the same time as t...
past_month_trouble_falling_asleep,a sleep outcome measured at the same time as t...
past_month_trouble_staying_asleep,a sleep outcome measured at the same time as t...
survey_weight,"a survey weight for population estimates, not ..."
household_id,an id with no meaning


In [4]:
print("Numeric features    :", NUMERIC)
print("Categorical features:", CATEGORICAL)

X = df[NUMERIC + CATEGORICAL]
y_class = df[TARGET_CLASS]
y_reg = df[TARGET_REG]

# Every column is either a feature, a target or deliberately excluded
assert set(X.columns) | set(EXCLUDED) == set(df.columns)
print(X.shape)

Numeric features    : ['age', 'weekly_work_hours', 'activity_guideline_status', 'anxiety_frequency', 'depression_frequency', 'serious_distress', 'kids_household', 'family_income', 'food_security', 'housing_cost', 'high_bloodpressure', 'had_asthma', 'had_arthritis', 'had_diabetes']
Categorical features: ['sex', 'education_level']
(1085, 16)


In [5]:
# Missing values per feature. They are filled in inside the pipeline, using the training data only.
missing = X.isna().sum()
missing[missing > 0].rename("missing values")

activity_guideline_status     4
anxiety_frequency             1
depression_frequency          1
serious_distress              4
food_security                 4
housing_cost                 11
had_asthma                    1
had_arthritis                 1
sex                           1
Name: missing values, dtype: int64

## 3.4 Preprocessing pipeline

`make_pipeline(model)` puts the same preprocessing in front of any model:

- **Numeric columns:** missing values become the column median, then the column is scaled to mean 0 and
  standard deviation 1. Scaling matters for k-NN and logistic regression, and does no harm to trees.
- **Categorical columns:** missing values become the most common category, then each category becomes
  its own 0/1 column (one-hot encoding).

Because the preprocessing is part of the pipeline, it learns medians, scales and categories from the
training data only when the pipeline is fitted. The test data never leaks into it.

In [6]:
def make_preprocessor():
    """Fill in missing values, scale numeric columns and one-hot encode categorical columns."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ])


def make_pipeline(model):
    """Return a pipeline with the shared preprocessing followed by `model`."""
    return Pipeline([("preprocess", make_preprocessor()), ("model", model)])

In [7]:
# Check that the preprocessing works: no missing values remain, and every column has a readable name.
# (Fitted on all rows only to inspect the output. The models in the next sections fit on training data.)
preview = make_preprocessor().fit(X)
transformed = preview.transform(X)
names = preview.get_feature_names_out()

print("Shape after preprocessing:", transformed.shape)
print("Missing values after preprocessing:", int(np.isnan(transformed).sum()))
print(list(names))

Shape after preprocessing: (1085, 26)
Missing values after preprocessing: 0
['numeric__age', 'numeric__weekly_work_hours', 'numeric__activity_guideline_status', 'numeric__anxiety_frequency', 'numeric__depression_frequency', 'numeric__serious_distress', 'numeric__kids_household', 'numeric__family_income', 'numeric__food_security', 'numeric__housing_cost', 'numeric__high_bloodpressure', 'numeric__had_asthma', 'numeric__had_arthritis', 'numeric__had_diabetes', 'categorical__sex_1.0', 'categorical__sex_2.0', 'categorical__education_level_1.0', 'categorical__education_level_2.0', 'categorical__education_level_3.0', 'categorical__education_level_4.0', 'categorical__education_level_5.0', 'categorical__education_level_6.0', 'categorical__education_level_7.0', 'categorical__education_level_8.0', 'categorical__education_level_9.0', 'categorical__education_level_10.0']


In [8]:
# The same function works for any model. A quick check that a full pipeline can be fitted and used:
check = make_pipeline(LogisticRegression(max_iter=1000)).fit(X, y_class)
print("Predicted probabilities for the first three students:", check.predict_proba(X.head(3))[:, 1].round(2))

Predicted probabilities for the first three students: [0.17 0.26 0.14]


**Summary.** The models get 16 input columns (14 numeric, 2 categorical), which become 26 columns after
one-hot encoding. The sleep outcomes, the survey weight and the id are left out on purpose. Next: split the data
into training and test sets (#18), then train the baseline and the models.